# Droplet number and aerosol size distributions at every site

For each site with collocated droplet number (Nd) and a particle number size distribution (PNSD) this notebook:

1. plots **time series** of Nd, N(D > 100 nm) and the monthly size distribution;
2. computes the **median size distribution with its interquartile range (IQR)**, for all hours and for the cloudy hours that enter the susceptibility fits (cloud fraction ≥ 0.5, cloud base ≤ 2 km, cloud top ≤ 3 km);
3. writes the medians and IQRs to CSV and the figures to PNG in `results/pnsd_summary/`.

**Run it from the `ARM_Data` folder** in the `arm` environment (`conda activate arm`, then `jupyter lab`). If Jupyter is missing: `conda install -n arm -c conda-forge jupyterlab`.

The droplet number and the cloudy-hour selection come from the built files in `results/<site>/`. The full PNSD record is read from the raw files in `data/`. It is averaged to hours, interpolated onto one log-spaced grid (10–1000 nm, 40 bins per decade) and cached in `results/pnsd_cache/`. The first run is slow for the big ARM records (ENA UHSAS is about 3,000 files, 23 GB); later runs read the cache in seconds. Sites whose built file or raw data are missing are skipped, so rerun the notebook after new sites finish.

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore", message="All-NaN slice")
warnings.filterwarnings("ignore", message="Mean of empty slice")
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

ROOT = Path.cwd()                      # the ARM_Data folder
sys.path.insert(0, str(ROOT))
from pnsd_tools import (GRID_NM, hourly_pnsd_arm, hourly_pnsd_ebas, hourly_pnsd_sems,
                        load_or_build, n_above, nd_hours)

CACHE = ROOT / "results" / "pnsd_cache"
OUT = ROOT / "results" / "pnsd_summary"
OUT.mkdir(parents=True, exist_ok=True)

SCREEN = dict(min_cloud_frac=0.5, max_base=2000.0, max_top=3000.0)   # same as the quicklooks
REBUILD = False                      # True re-reads all raw PNSD files
MIN_HOURS_PER_MONTH = 24             # months with fewer hours are left blank

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
C_ALL, C_CLOUD = "#3B6FB6", "#D9822B"      # all hours / cloudy (matched) hours

## Sites

One row per droplet-number + aerosol pairing. `cache` names the hourly PNSD file: rows that share an aerosol record (SGP NDROP and SGP Cloudnet) share the cache. `caution` marks Nd records we decided not to trust in absolute terms.

In [2]:
E = lambda site: sorted(str(p) for p in (ROOT / "data" / "ebas" / site).glob("*.nc"))
A = lambda *p: str(ROOT.joinpath("data", "arm", *p))
R = lambda *p: ROOT.joinpath("results", *p)

SITES = [
  # key, label, Nd source, built file, PNSD kind, PNSD source, cache, caution
  dict(key="ena_uhsas", label="ENA (UHSAS)", nd="NDROP", built=R("ena", "ena_2014_2023_uhsas.nc"),
       kind="arm", src=A("ena", "uhsas"), cache="ena_uhsas"),
  dict(key="ena_smps", label="ENA (SMPS)", nd="NDROP", built=R("ena", "ena_2022_2026_smps.nc"),
       kind="arm", src=A("ena", "smps"), cache="ena_smps"),
  dict(key="sgp_smps", label="SGP (SMPS)", nd="NDROP", built=R("sgp", "sgp_2016_2025_smps.nc"),
       kind="arm", src=A("sgp", "smps"), cache="sgp_smps"),
  dict(key="sgp_cloudnet", label="SGP (Cloudnet Nd)", nd="Cloudnet", built=R("sgp", "sgp_cloudnet.nc"),
       kind="arm", src=A("sgp", "smps"), cache="sgp_smps", caution="Cloudnet Nd ~1.8x NDROP here"),
  dict(key="lasic_smps", label="LASIC (SMPS)", nd="NDROP", built=R("lasic", "lasic_2016_2017_smps.nc"),
       kind="arm", src=A("lasic", "smps"), cache="lasic_smps", caution="NDROP biased high"),
  dict(key="lasic_uhsas", label="LASIC (UHSAS)", nd="NDROP", built=R("lasic", "lasic_2016_2017_uhsas.nc"),
       kind="arm", src=A("lasic", "uhsas"), cache="lasic_uhsas", caution="NDROP biased high"),
  dict(key="epc_pier", label="EPCAPE pier (SMPS)", nd="NDROP", built=R("epcape_pier", "epc_2023_2024_smps.nc"),
       kind="arm", src=A("epc", "smps"), cache="epc_smps"),
  dict(key="epc_soledad", label="EPCAPE Mt Soledad (FM-120 + SEMS)", nd="FM-120 in situ",
       built=R("epcape_soledad", "epcape_soledad_hourly.nc"), kind="sems",
       src=str(ROOT / "data" / "epcape_soledad" / "sems"), logs=str(ROOT / "data" / "epcape_soledad" / "sems_logs"),
       cache="epc_soledad_sems", caution="cloudy-hour PNSD is the same-hour SEMS (may be in cloud)"),
  dict(key="hyytiala", label="Hyytiälä", nd="Cloudnet", built=R("hyytiala", "hyytiala_cloudnet.nc"),
       kind="ebas", src=[str(ROOT / "data" / "ebas" / "FI0050R_dmps_2013_2026.nc")], cache="hyytiala_dmps"),
  dict(key="mace-head", label="Mace Head", nd="Cloudnet", built=R("mace-head", "mace-head_cloudnet.nc"),
       kind="ebas", src=E("mace-head"), cache="mace-head_smps", caution="Cloudnet Nd above total N"),
  dict(key="cabauw", label="Cabauw", nd="Cloudnet", built=R("cabauw", "cabauw_cloudnet.nc"),
       kind="ebas", src=E("cabauw"), cache="cabauw_mpss"),
  dict(key="granada", label="Granada", nd="Cloudnet", built=R("granada", "granada_cloudnet.nc"),
       kind="ebas", src=E("granada"), cache="granada_smps"),
  dict(key="palaiseau", label="Palaiseau", nd="Cloudnet", built=R("palaiseau", "palaiseau_cloudnet.nc"),
       kind="ebas", src=E("palaiseau"), cache="palaiseau_smps"),
  dict(key="schneefernerhaus", label="Schneefernerhaus", nd="Cloudnet",
       built=R("schneefernerhaus", "schneefernerhaus_cloudnet.nc"),
       kind="ebas", src=E("schneefernerhaus"), cache="schneefernerhaus_smps", caution="Cloudnet Nd too high"),
  # in progress: picked up automatically once built
  dict(key="ny-alesund", label="Ny-Ålesund (Zeppelin DMPS)", nd="Cloudnet", built=R("ny-alesund", "ny-alesund_cloudnet.nc"),
       kind="ebas", src=E("ny-alesund"), cache="zeppelin_dmps"),
  dict(key="kenttarova", label="Kenttärova (Pallas DMPS)", nd="Cloudnet", built=R("kenttarova", "kenttarova_cloudnet.nc"),
       kind="ebas", src=E("kenttarova"), cache="pallas_dmps"),
  dict(key="oliktok", label="Oliktok (UHSAS)", nd="Cloudnet", built=R("oliktok", "oliktok_cloudnet.nc"),
       kind="arm", src=A("oli", "uhsas"), cache="oli_uhsas"),
  dict(key="bankhead", label="Bankhead (SMPS)", nd="Cloudnet", built=R("bankhead", "bankhead_cloudnet.nc"),
       kind="arm", src=A("bnf", "smps"), cache="bnf_smps"),
]

## Load Nd and the hourly PNSD

This is the slow cell on the first run. It prints progress for each ARM folder.

In [3]:
def builder(s):
    if s["kind"] == "arm":
        return lambda: hourly_pnsd_arm(s["src"])
    if s["kind"] == "ebas":
        return lambda: hourly_pnsd_ebas(s["src"])
    return lambda: hourly_pnsd_sems(s["src"], s.get("logs"))

def has_raw(s):
    if s["kind"] == "ebas":
        return len(s["src"]) > 0 and all(os.path.exists(p) for p in s["src"])
    return os.path.isdir(s["src"]) and any(os.scandir(s["src"]))

DATA, _pnsd = {}, {}
for s in SITES:
    if not Path(s["built"]).exists():
        print(f"skip {s['label']:<36} no built file yet ({Path(s['built']).relative_to(ROOT)})")
        continue
    cache = CACHE / f"{s['cache']}_pnsd_hourly.nc"
    if s["cache"] not in _pnsd:
        if not cache.exists() and not has_raw(s):
            print(f"skip {s['label']:<36} raw PNSD not found")
            continue
        print(f"PNSD {s['label']:<36} {'cache' if cache.exists() and not REBUILD else 'reading raw files ...'}")
        _pnsd[s["cache"]] = load_or_build(str(cache), builder(s), rebuild=REBUILD)
    with xr.open_dataset(s["built"]) as ds:
        hrs = nd_hours(ds, **SCREEN)
    DATA[s["key"]] = dict(meta=s, pnsd=_pnsd[s["cache"]], nd=hrs)
print(f"\n{len(DATA)} sites loaded")

PNSD ENA (UHSAS)                          cache
PNSD ENA (SMPS)                           cache
PNSD SGP (SMPS)                           cache
PNSD LASIC (SMPS)                         reading raw files ...
  smps: 200/518 files
  smps: 400/518 files
  smps: 518/518 files
PNSD LASIC (UHSAS)                        reading raw files ...
  uhsas: 200/561 files
  uhsas: 400/561 files
  uhsas: 561/561 files
PNSD EPCAPE pier (SMPS)                   reading raw files ...
  smps: 200/363 files
  smps: 363/363 files
PNSD EPCAPE Mt Soledad (FM-120 + SEMS)    reading raw files ...
PNSD Hyytiälä                             reading raw files ...
PNSD Mace Head                            reading raw files ...
PNSD Cabauw                               reading raw files ...
PNSD Granada                              reading raw files ...
PNSD Palaiseau                            reading raw files ...
PNSD Schneefernerhaus                     reading raw files ...
PNSD Ny-Ålesund (Zeppelin DMPS)      

## Overview

In [ ]:
def q(x):
    x = pd.Series(x).dropna()
    return f"{x.median():.0f} [{x.quantile(.25):.0f}–{x.quantile(.75):.0f}]" if len(x) else "–"

rows = []
for k, d in DATA.items():
    s, p, h = d["meta"], d["pnsd"], d["nd"]
    meas = p.columns[p.notna().mean() > 0.2]
    rows.append({"site": s["label"], "Nd source": s["nd"],
                 "cloudy hours": len(h),
                 "period (cloudy)": f"{h.index.min():%Y-%m}–{h.index.max():%Y-%m}" if len(h) else "–",
                 "Nd median [IQR] cm-3": q(h["Nd"]),
                 "N>100 cloudy hours": q(h["N100"]),
                 "N>100 all hours": q(n_above(p, 100)),
                 "PNSD hours": len(p),
                 "PNSD range nm": f"{meas.min():.0f}–{meas.max():.0f}" if len(meas) else "–",
                 "caution": s.get("caution", "")})
overview = pd.DataFrame(rows).set_index("site")
overview.to_csv(OUT / "site_overview.csv")
overview

,Nd source,cloudy hours,period (cloudy),Nd median [IQR] cm-3,N>100 cloudy hours,N>100 all hours,PNSD hours,PNSD range nm,caution
site,,,,,,,,,
ENA (UHSAS),NDROP,2308,2015-08–2023-12,80 [40–164],128 [79–196],125 [73–197],60834,56–944,
ENA (SMPS),NDROP,615,2022-11–2025-07,57 [28–131],132 [85–208],128 [80–204],28402,11–473,
SGP (SMPS),NDROP,2257,2016-11–2025-05,173 [91–333],478 [301–750],641 [360–994],73569,11–501,
SGP (Cloudnet Nd),Cloudnet,776,2020-01–2025-12,674 [347–1114],388 [255–629],641 [360–994],73569,11–501,Cloudnet Nd ~1.8x NDROP here
LASIC (SMPS),NDROP,1900,2016-07–2017-09,219 [114–434],129 [78–185],137 [85–211],11743,12–447,NDROP biased high
LASIC (UHSAS),NDROP,2027,2016-07–2017-09,217 [115–423],137 [85–198],148 [88–219],13137,56–944,NDROP biased high
EPCAPE pier (SMPS),NDROP,633,2023-03–2024-01,239 [138–415],438 [304–534],455 [284–693],8389,11–473,
EPCAPE Mt Soledad (FM-120 + SEMS),FM-120 in situ,138,2023-03–2024-02,228 [143–309],187 [67–364],264 [127–427],7795,11–794,cloudy-hour PNSD is the same-hour SEMS (may be...
Hyytiälä,Cloudnet,893,2018-01–2025-12,162 [91–300],221 [126–402],302 [154–599],102901,10–1000,


: 

In [3]:
import time, matplotlib, pandas
print(matplotlib.__version__, pandas.__version__, matplotlib.get_backend())
t = time.time(); fig, ax = plt.subplots(); ax.plot([1, 2]); fig.savefig(OUT / "test.png"); plt.close(fig)
print(f"simple plot {time.time() - t:.1f} s")
t = time.time(); plot_timeseries(list(DATA.values())[0]); print(f"ENA time series {time.time() - t:.1f} s")

: 

## Time series

For each site:

- **Nd** in the cloudy hours (dots) with the monthly median and IQR;
- **N(D > 100 nm)** for all hours (monthly median and IQR, blue) and for the cloudy hours (orange);
- the **monthly median size distribution** (colour: dN/dlogDp, log scale);
- the number of cloudy hours per month.

Months with fewer than `MIN_HOURS_PER_MONTH` hours are left blank.

In [ ]:
def monthly(x, min_n=MIN_HOURS_PER_MONTH):
    g = x.resample("MS")
    out = pd.DataFrame({"med": g.median(), "q25": g.quantile(.25), "q75": g.quantile(.75), "n": g.count()})
    out.loc[out["n"] < min_n, ["med", "q25", "q75"]] = np.nan
    return out

def band(ax, m, color, label):
    t = m.index + pd.Timedelta(days=15)
    ax.fill_between(t, m["q25"], m["q75"], color=color, alpha=0.25, lw=0, step=None)
    ax.plot(t, m["med"], color=color, lw=1.6, label=label)

def plot_timeseries(d):
    s, p, h = d["meta"], d["pnsd"], d["nd"]
    n100_all = n_above(p, 100)
    t0 = min(h.index.min(), p.index.min()) if len(h) else p.index.min()
    t1 = max(h.index.max(), p.index.max()) if len(h) else p.index.max()
    fig, ax = plt.subplots(4, 1, figsize=(12, 10.5), sharex=True,
                           gridspec_kw=dict(height_ratios=[1, 1, 1.1, 0.45]))
    # Nd
    ax[0].plot(h.index, h["Nd"], ".", ms=2, color="0.6", alpha=0.5, label="cloudy hours")
    band(ax[0], monthly(h["Nd"], 5), C_CLOUD, "monthly median (IQR shaded)")
    ax[0].set_yscale("log"); ax[0].set_ylabel(f"Nd ({s['nd']}), cm$^{{-3}}$")
    ax[0].legend(loc="upper left", fontsize=8, frameon=False)
    # N100
    band(ax[1], monthly(n100_all), C_ALL, "all hours")
    band(ax[1], monthly(h["N100"], 5), C_CLOUD, "cloudy hours")
    ax[1].set_yscale("log"); ax[1].set_ylabel("N(D > 100 nm), cm$^{-3}$")
    ax[1].legend(loc="upper left", fontsize=8, frameon=False)
    # monthly PNSD (sequential colormap: dN/dlogDp is positive)
    pm = p.resample("MS").median()
    pm[p.resample("MS").count().max(axis=1) < MIN_HOURS_PER_MONTH] = np.nan
    edges_t = mdates.date2num(list(pm.index) + [pm.index[-1] + pd.offsets.MonthBegin(1)]) if len(pm) else []
    lg = np.log10(GRID_NM); step = lg[1] - lg[0]
    edges_d = 10 ** np.r_[lg - step / 2, lg[-1] + step / 2]
    z = pm.to_numpy().T
    if np.isfinite(z).any():
        pos = z[np.isfinite(z) & (z > 0)]
        im = ax[2].pcolormesh(edges_t, edges_d, np.ma.masked_invalid(np.where(z > 0, z, np.nan)), cmap="viridis",
                              norm=LogNorm(np.percentile(pos, 2), np.percentile(pos, 99.5)), shading="flat")
        fig.colorbar(im, ax=ax[2], pad=0.01, fraction=0.03, label="dN/dlogDp, cm$^{-3}$")
    ax[2].set_yscale("log"); ax[2].set_ylabel("Diameter, nm"); ax[2].grid(False)
    ax[2].axhline(100, color="w", lw=0.8, ls="--")
    # hours per month
    cnt = h["Nd"].resample("MS").count()
    ax[3].bar(cnt.index, cnt.values, width=25, align="edge", color=C_CLOUD)
    ax[3].set_ylabel("cloudy h / month")
    ax[3].set_xlim(t0 - pd.Timedelta(days=15), t1 + pd.Timedelta(days=15))
    title = f"{s['label']}: {len(h)} cloudy hours"
    if s.get("caution"):
        title += f"  ({s['caution']})"
    ax[0].set_title(title, loc="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT / f"{s['key']}_timeseries.png", dpi=150)
    plt.show()

for d in DATA.values():
    plot_timeseries(d)

## Median size distribution with IQR

Blue: all hours of the PNSD record. Orange: the cloudy hours that enter the susceptibility fits. Shading is the 25–75th percentile at each diameter. The medians and IQRs are written to `results/pnsd_summary/<site>_pnsd_median_iqr.csv`.

In [ ]:
def stats(p):
    a = p.to_numpy()
    with np.errstate(all="ignore"):
        ok = np.isfinite(a).mean(axis=0) >= 0.2          # diameters measured in >= 20% of hours
        out = pd.DataFrame({"median": np.nanmedian(a, axis=0), "q25": np.nanpercentile(a, 25, axis=0),
                            "q75": np.nanpercentile(a, 75, axis=0), "n_hours": np.isfinite(a).sum(axis=0)},
                           index=pd.Index(GRID_NM, name="diameter_nm"))
    out.loc[~ok, ["median", "q25", "q75"]] = np.nan
    return out

STATS = {}
for k, d in DATA.items():
    p, h = d["pnsd"], d["nd"]
    cloudy = p.reindex(h.index.intersection(p.index))
    st_all, st_cld = stats(p), stats(cloudy) if len(cloudy) else None
    STATS[k] = (st_all, st_cld, len(p), len(cloudy))
    t = st_all.add_prefix("all_")
    if st_cld is not None:
        t = t.join(st_cld.add_prefix("cloudy_"))
    t.to_csv(OUT / f"{k}_pnsd_median_iqr.csv", float_format="%.4g")

n = len(STATS); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow), squeeze=False)
for ax, (k, (sa, sc, na, nc)) in zip(axs.flat, STATS.items()):
    for st, col, lab in ((sa, C_ALL, f"all hours ({na:,})"), (sc, C_CLOUD, f"cloudy hours ({nc:,})")):
        if st is None:
            continue
        ax.fill_between(st.index, st["q25"], st["q75"], color=col, alpha=0.25, lw=0)
        ax.plot(st.index, st["median"], color=col, lw=1.6, label=lab)
    ax.set_xscale("log"); ax.set_xlim(10, 1000); ax.set_ylim(bottom=0)
    ax.set_title(DATA[k]["meta"]["label"], loc="left", fontsize=10)
    ax.legend(fontsize=7, frameon=False, loc="upper right")
    ax.set_xlabel("Diameter, nm"); ax.set_ylabel("dN/dlogDp, cm$^{-3}$")
for ax in list(axs.flat)[n:]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "median_pnsd_iqr_all_sites.png", dpi=150)
plt.show()

## All sites on one axis

Median size distribution in the cloudy hours (log scale, so the marine and continental sites fit on one plot). Dashed lines are sites flagged with a caution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
cols = plt.cm.tab20(np.linspace(0, 1, 20))
for i, (k, (sa, sc, na, nc)) in enumerate(STATS.items()):
    st = sc if sc is not None and nc >= 20 else None
    if st is None:
        continue
    ls = "--" if DATA[k]["meta"].get("caution") else "-"
    ax.plot(st.index, st["median"], ls, color=cols[i % 20], lw=1.6, label=f"{DATA[k]['meta']['label']} ({nc:,} h)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(10, 1000); ax.set_ylim(1, None)
ax.set_xlabel("Diameter, nm"); ax.set_ylabel("dN/dlogDp, cm$^{-3}$")
ax.set_title("Median PNSD in cloudy hours", loc="left")
ax.legend(fontsize=7, frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
fig.savefig(OUT / "median_pnsd_cloudy_overlay.png", dpi=150, bbox_inches="tight")
plt.show()

## Notes

- **Nd sources differ.** NDROP (ENA, SGP, LASIC, EPCAPE pier) is daytime overcast only. Cloudnet N_scaled has a site-dependent bias (about 1.8× NDROP at SGP). Mt Soledad is in-situ FM-120. Compare Nd levels across sources with care.
- **Instrument ranges differ.** UHSAS starts at about 60 nm; the SMPS and DMPS reach down to 10 nm. Blank diameters are outside the measured range.
- **N(D > 100 nm) for all hours** is integrated from the gridded PNSD up to the instrument's top size. For the cloudy hours it is taken from the built files, which are the values used in the susceptibility fits (the two agree to within ~0.5% where they overlap).
- **EPCAPE Mt Soledad:** the built file pairs Nd with clear-air SEMS scans within ±3 h. The cloudy-hour PNSD here is the same-hour SEMS, which can be sampled in cloud.
- Set `REBUILD = True` after adding raw files to a site that is already cached.